Each preset corresponds to a different target microwave operating point (4–8 GHz), 
meaning the plasma density pattern across the array is different in each case. 
The audio spectrum gives us a time-varying set of mixing weights, and we interpolate between those operating points while rate-limiting the voltage changes to keep the discharge stable.

In [ ]:
#run this to see if PMM working

import sys
import os
# Add the parent folder (PMM-Design) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from PMM.PMMInSitu import PMMInSitu
PMM = PMMInSitu('../confs/conf_test.yaml')

# # Run these 2 lines:
# PMM.Set_Bulb_VI('all', 10, 2) #set bulbs(all, volts, amps)
# PMM.Activate_Bulb('all')

# OR run these 2 lines:
# PMM.Config_Check()
PMM.Deactivate_Bulb('all')

In [2]:
import os, sys
import numpy as np
import time
from scipy.io import wavfile

# --- paths ---
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(REPO_ROOT)

ASSET_DIR  = os.path.join(REPO_ROOT, "show_assets")
PRESET_DIR = ASSET_DIR

AUDIO_WAV = os.path.join(ASSET_DIR, "Advent Chamber Orchestra - Mozart - Eine Kleine Nachtmusik allegro - Free Music Archive (CC BY-SA).wav") #### update file name
PRESET_FILES = [os.path.join(PRESET_DIR, f"rho_preset{i}GHz.csv") for i in range(4, 9)] # update file names

print("cwd:", os.getcwd())
print("REPO_ROOT:", REPO_ROOT)
print("AUDIO_WAV exists?", os.path.exists(AUDIO_WAV))
print("Preset files exist?", [os.path.exists(f) for f in PRESET_FILES])

from PMM.PMMInSitu import PMMInSitu

CONF_FILE = os.path.join(REPO_ROOT, "confs", "conf_test.yaml")
CONF_DIR  = os.path.join(REPO_ROOT, "confs")  + os.sep # where VtoI.txt lives

PMM = PMMInSitu(CONF_FILE, conf_dir=CONF_DIR, verbose=True)

PMM.Deactivate_Bulb('all')


cwd: /Users/scsl/PMM-Design/scripts
REPO_ROOT: /Users/scsl/PMM-Design
AUDIO_WAV exists? True
Preset files exist? [True, True, True, True, True]
checking port /dev/tty.usbserial-B003LKT9
checking port /dev/tty.usbserial-B0032GMG
checking port /dev/tty.usbserial-B003L2CD
checking port /dev/tty.usbserial-B003LJKJ
checking port /dev/tty.usbserial-B0032GQ2
checking port /dev/tty.usbserial-B0019Q3M
checking port /dev/tty.usbserial-B0032389
checking port /dev/tty.usbserial-B001K6BU
checking port /dev/tty.usbserial-B0013LEA
checking port /dev/tty.usbserial-B00295EU


In [3]:
# ---- show settings ----
FPS = 2 # how long state holds before changing (15 FPS means ~66 ms per update...)
HOP = 1.0 / FPS
ONLY_VOLTAGE = False # experiment with this

# mapping params
FPM_GHZ = 18.0
K = 0.33
S = 1.0

# safety / smoothness
DV_MAX = 0.8          # max volts change per frame per bulb (ADJUST THIS IF TOO SUBTLE)
WARMUP_CYCLES = 3       
DUTY_CYCLE = 0.8       # warmup duty cycle (your function uses this)

# ignition / running levels
IGNITION_V = 14
IGNITION_I = 10
RUN_V = 10
RUN_I = 10

# dimness
DIM_V = 7.0      # pick a “barely on but stable” sustain level
DIM_I = 2.0      # if you write current too (recommended)
ENV_GAMMA = 2.5  # >1 makes quiet parts quieter + peaks pop more
ONLY_VOLTAGE = False  # important if you want real dimming

# hold settings
HOLD_S = 0.8        # how long to hold each chosen preset
WTA = True          # winner-take-all instead of blending
PRINT_SWITCHES = True



In [4]:
def five_band_weights(x, sr, hop_s):
    """
    x: mono audio float32 [-1,1], shape (N,)
    returns W shape (T,5), rows sum to 1
    """
    N = len(x)
    win = int(sr * hop_s)
    if win < 512:
        win = 512

    # 5 simple frequency bands in Hz (tweak if you want)
    bands = [(20,120), (120,300), (300,900), (900,3000), (3000,12000)]

    T = int(np.floor((N - win) / win)) + 1
    W = np.zeros((T, 5), dtype=float)

    for t in range(T):
        seg = x[t*win:(t+1)*win]
        seg = seg * np.hanning(len(seg))
        X = np.fft.rfft(seg)
        P = (np.abs(X) ** 2)
        freqs = np.fft.rfftfreq(len(seg), d=1/sr)

        for i, (f0, f1) in enumerate(bands):
            m = (freqs >= f0) & (freqs < f1)
            W[t, i] = np.sum(P[m])

    # normalize so each row sums to 1
    W += 1e-12
    W /= np.sum(W, axis=1, keepdims=True)

    # smooth weights over time (simple IIR)
    alpha = 0.25
    for t in range(1, T):
        W[t] = (1-alpha)*W[t-1] + alpha*W[t]
        W[t] /= np.sum(W[t])

    return W


In [5]:
# dimness helper 
def loudness_env(x, sr, hop_s):
    """RMS loudness per frame -> (T,) in [0,1]"""
    win = int(sr * hop_s)
    if win < 512: win = 512
    T = int(np.floor((len(x) - win) / win)) + 1
    e = np.zeros(T, float)

    for t in range(T):
        seg = x[t*win:(t+1)*win]
        e[t] = np.sqrt(np.mean(seg*seg) + 1e-12)  # RMS

    # normalize + mild smoothing
    e = e / (np.max(e) + 1e-12)
    alpha = 0.2
    for t in range(1, T):
        e[t] = (1-alpha)*e[t-1] + alpha*e[t]
    return e


In [ ]:
def load_rho_csv(path):
    r = np.loadtxt(path, delimiter=",")
    return np.asarray(r).reshape(-1)

def run_show(pmm):
    # ---- load presets (rho) ----
    rhos = [load_rho_csv(f) for f in PRESET_FILES]

    lens = [r.size for r in rhos]
    if len(set(lens)) != 1:
        raise ValueError(f"Preset rho sizes mismatch: {lens}")
    nbulbs = lens[0]

    # ---- convert presets to bulb setpoints (N,2) ----
    presets = [pmm.Rho_to_Bulb(r, pmm.f_a(FPM_GHZ), knob=K, scale=S) for r in rhos]
    presets = [np.asarray(ps, float) for ps in presets]

    for i, ps in enumerate(presets):
        if ps.shape != (nbulbs, 2):
            raise ValueError(f"Preset {i} bulbset has shape {ps.shape}, expected {(nbulbs,2)}")

    # ---- audio load ----
    sr, x = wavfile.read(AUDIO_WAV)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = x.astype(np.float32)
    x /= (np.max(np.abs(x)) + 1e-12)

    # ---- compute band weights + loudness envelope ----
    W = five_band_weights(x, sr, HOP)        # (T_w, 5)
    env = loudness_env(x, sr, HOP)           # (T_e,)
    env = env ** ENV_GAMMA                   # more contrast

    T = min(W.shape[0], env.shape[0])        # safety: match lengths

    # ---- warmup + ignite once ----
    pmm.Config_Warmup(T=WARMUP_CYCLES, ballasts="New", duty_cycle=DUTY_CYCLE)
    pmm.Set_Bulb_VI('all', IGNITION_V, IGNITION_I, verbose=False)
    pmm.Activate_Bulb('all')
    time.sleep(1.0)
    pmm.Set_Bulb_VI('all', RUN_V, RUN_I, verbose=False)

    # start from preset 0
    current = np.array(presets[0], float, copy=True)

    # ---- main loop ----
    t0 = time.time()
    try:
        for ti in range(T):
            # mix presets using weights
            target = np.zeros_like(current)
            for bi in range(5):
                target += W[ti, bi] * presets[bi]

            # limit per-frame voltage step
            target = pmm.limit_step(current, target, dV_max=DV_MAX)

            # apply global dimming envelope toward (DIM_V, DIM_I)
            e = float(env[ti])  # 0..1

            target_env = target.copy()
            target_env[:, 0] = DIM_V + e * (target[:, 0] - DIM_V)  # voltage
            target_env[:, 1] = DIM_I + e * (target[:, 1] - DIM_I)  # current

            # write to hardware
            pmm.Apply_BulbSet(target_env, only_voltage=ONLY_VOLTAGE)
            current = target_env

            # 5) pacing
            next_time = t0 + (ti + 1) * HOP
            sleep = next_time - time.time()
            if sleep > 0:
                time.sleep(sleep)

    finally:
        pmm.Deactivate_Bulb('all')


In [7]:
run_show(PMM)


[Scale_Rho_fp] size=91 rho[min,med,max]=(0,0.741,1.29) fp_GHz[min,med,max]=(0,10.33,17.99) clipped_hi=0 clipped_lo=0 ceil=20.0 GHz
[Scale_Rho_fp] size=91 rho[min,med,max]=(0,0.642,1.29) fp_GHz[min,med,max]=(0,8.952,17.99) clipped_hi=0 clipped_lo=0 ceil=20.0 GHz
[Scale_Rho_fp] size=91 rho[min,med,max]=(0,0.667,1.29) fp_GHz[min,med,max]=(0,9.301,17.99) clipped_hi=0 clipped_lo=0 ceil=20.0 GHz
[Scale_Rho_fp] size=91 rho[min,med,max]=(0,0.742,1.29) fp_GHz[min,med,max]=(0,10.35,17.99) clipped_hi=0 clipped_lo=0 ceil=20.0 GHz
[Scale_Rho_fp] size=91 rho[min,med,max]=(0,0.748,1.29) fp_GHz[min,med,max]=(0,10.43,17.99) clipped_hi=0 clipped_lo=0 ceil=20.0 GHz
Warmup cycle 1
Warmup cycle 2
Warmup cycle 3


KeyboardInterrupt: 

In [ ]:
#### tests